In [1]:
import warnings
import pandas as pd, numpy as np, joblib
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

K = 20                                                   # flows per bundle
CIC_DIR = Path("../data/MachineLearningCSV/TrafficLabelling")
CAPTURES = Path("../data/captures")
MODEL_PATH = Path("../models/rf_window_v1.joblib")

In [2]:
NEED = ["Source IP", "Destination IP", "Destination Port", "Timestamp",
        "Total Fwd Packets", "Total Backward Packets", "Label"]

def family(label):
    l = str(label).strip()
    if l == "BENIGN": return "benign"
    if l == "PortScan": return "portscan"
    if l in ("FTP-Patator", "SSH-Patator"): return "bruteforce"
    if l.startswith("DoS") or l == "DDoS": return "dos"
    if l.startswith("Web Attack"): return "web"
    return "other"

parts = []
for f in sorted(CIC_DIR.glob("*.csv")):
    cols = pd.read_csv(f, nrows=0, encoding="latin1").columns
    orig = {c.strip(): c for c in cols}
    d = pd.read_csv(f, encoding="latin1", usecols=[orig[c] for c in NEED])
    d.columns = [c.strip() for c in d.columns]
    d["file"] = f.name
    parts.append(d)
    print(f.name, len(d))

flows = pd.concat(parts, ignore_index=True).dropna()
flows["family"] = flows["Label"].map(family)
flows = flows[flows["family"] != "other"].reset_index(drop=True)
print(flows.shape)

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv 225745
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv 286467
Friday-WorkingHours-Morning.pcap_ISCX.csv 191033
Monday-WorkingHours.pcap_ISCX.csv 529918
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv 288602
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv 458968
Tuesday-WorkingHours.pcap_ISCX.csv 445909
Wednesday-workingHours.pcap_ISCX.csv 692703
(2828730, 9)


In [3]:
def chunk_features(df, keys):
    """One row per bundle of exactly K flows from the same source."""
    g = df.groupby(keys, sort=False)
    w = g.agg(n=("dst_port", "size"), n_dst_ports=("dst_port", "nunique"),
              n_dst_hosts=("dst_ip", "nunique"), no_reply_share=("no_reply", "mean"),
              mean_fwd_pkts=("fwd", "mean"), mean_bwd_pkts=("bwd", "mean"))
    top = df.groupby(keys + ["dst_port"], sort=False).size().groupby(level=keys, sort=False).max()
    w["top_port_share"] = top / w["n"]
    return w[w["n"] == K]

f2 = flows.rename(columns={"Source IP": "src_ip", "Destination IP": "dst_ip",
                           "Destination Port": "dst_port",
                           "Total Fwd Packets": "fwd", "Total Backward Packets": "bwd"})
f2["dst_port"] = pd.to_numeric(f2["dst_port"], errors="coerce")
f2 = f2.dropna(subset=["dst_port"]).copy()
f2["no_reply"] = (f2["bwd"] == 0).astype(float)
f2["chunk"] = f2.groupby(["file", "src_ip"], sort=False).cumcount() // K

w = chunk_features(f2, ["file", "src_ip", "chunk"])
fam = f2.groupby(["file", "src_ip", "chunk"], sort=False)["family"].agg(lambda s: s.mode().iat[0])
w = w.join(fam).reset_index()

# earlier bundles of each source are for training, the last 30% for testing
grp = w.groupby(["file", "src_ip"], sort=False)["chunk"]
w["is_test"] = (grp.rank(method="first") / grp.transform("size")) > 0.7
print(w.shape)
w.groupby(["family", "is_test"]).size().unstack(fill_value=0)

(132494, 12)


is_test,False,True
family,,
benign,71371,33228
bruteforce,490,211
dos,13404,5735
portscan,5562,2384
web,77,32


In [4]:
FEATS = ["n_dst_ports", "n_dst_hosts", "top_port_share",
         "no_reply_share", "mean_fwd_pkts", "mean_bwd_pkts"]
assert not {"family", "file", "src_ip", "chunk", "is_test", "n"} & set(FEATS), "label or ID column in FEATS"
tr, te = w[~w["is_test"]], w[w["is_test"]]

rfw = RandomForestClassifier(n_estimators=200, min_samples_leaf=2,
                             class_weight="balanced", n_jobs=-1, random_state=42)
rfw.fit(tr[FEATS], tr["family"])
pred = rfw.predict(te[FEATS])
print(classification_report(te["family"], pred, digits=4, zero_division=0))
labels = sorted(w["family"].unique())
print(pd.DataFrame(confusion_matrix(te["family"], pred, labels=labels), index=labels, columns=labels))
print(pd.Series(rfw.feature_importances_, index=FEATS).sort_values(ascending=False).round(4))

ben = (te["family"] == "benign").to_numpy()
flagged = (pred != "benign")
print(f"\nBundle-level false alarm rate: {(flagged & ben).sum() / ben.sum():.2%} "
      f"({(flagged & ben).sum()} of {ben.sum()} benign bundles)")
print(f"Attack detection rate (any attack flagged as an attack): "
      f"{(flagged & ~ben).sum() / (~ben).sum():.2%} ({(flagged & ~ben).sum()} of {(~ben).sum()} attack bundles)")

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({"model": rfw, "features": FEATS}, MODEL_PATH)

              precision    recall  f1-score   support

      benign     0.9797    0.9918    0.9857     33228
  bruteforce     0.9688    0.5877    0.7316       211
         dos     0.9950    0.9013    0.9458      5735
    portscan     0.9026    0.9878    0.9433      2384
         web     0.9500    0.5938    0.7308        32

    accuracy                         0.9767     41590
   macro avg     0.9592    0.8125    0.8674     41590
weighted avg     0.9773    0.9767    0.9763     41590

            benign  bruteforce   dos  portscan  web
benign       32955           4    15       254    0
bruteforce      87         124     0         0    0
dos            565           0  5169         0    1
portscan        29           0     0      2355    0
web              2           0    11         0   19
mean_fwd_pkts     0.3057
mean_bwd_pkts     0.2644
n_dst_ports       0.1535
top_port_share    0.1274
n_dst_hosts       0.0966
no_reply_share    0.0524
dtype: float64

Bundle-level false alarm rate: 0.

['..\\models\\rf_window_v1.joblib']

In [5]:
def zeek_windows(conn):
    d = conn.copy()
    d["ts"] = pd.to_numeric(d["ts"]); d = d.sort_values("ts")
    for c in ["orig_pkts", "resp_pkts"]:
        if c not in d.columns: d[c] = np.nan
        d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0)
    d["dst_port"] = pd.to_numeric(d["id.resp_p"], errors="coerce")
    d = d.rename(columns={"id.orig_h": "src_ip", "id.resp_h": "dst_ip",
                          "orig_pkts": "fwd", "resp_pkts": "bwd"})
    d["no_reply"] = (d["bwd"] == 0).astype(float)
    d = d.dropna(subset=["dst_port"]).copy()
    d["chunk"] = d.groupby("src_ip", sort=False).cumcount() // K
    return chunk_features(d, ["src_ip", "chunk"])

In [6]:
bundle = joblib.load(MODEL_PATH)
rfw, FEATS = bundle["model"], bundle["features"]

KINDS = ["benign", "bruteforce", "portscan", "dos", "web"]
rows = []
for f in sorted(CAPTURES.rglob("conn.log")):
    conn = pd.read_json(f, lines=True)
    name = f.parent.name
    if len(conn) < 50:
        print("skipping tiny capture:", name, len(conn)); continue
    ww = zeek_windows(conn)
    if len(ww) == 0:
        print("no full 20-flow bundles in", name); continue
    p = pd.Series(rfw.predict(ww[FEATS]))
    expected = next((k for k in KINDS if k in name), None)
    rows.append({"capture": name, "flows": len(conn), "bundles": len(ww),
                 "expected": expected,
                 "pct_predicted_as_expected": round((p == expected).mean() * 100, 1),
                 "predictions": p.value_counts().to_dict()})
pd.DataFrame(rows)

skipping tiny capture: zeek_benign_01 3
skipping tiny capture: zeek_bruteforce_ssh_01 3


,capture,flows,bundles,expected,pct_predicted_as_expected,predictions
0,zeek_benign_02,235,11,benign,100.0,{'benign': 11}
1,zeek_benign_03,3012,150,benign,100.0,{'benign': 150}
2,zeek_bruteforce_02,323,16,bruteforce,0.0,{'benign': 16}
3,zeek_dos_01,1000,50,dos,100.0,{'dos': 50}
4,zeek_portscan_01,65554,3277,portscan,99.8,"{'portscan': 3272, 'benign': 5}"
5,zeek_web_01,90,4,web,0.0,{'benign': 4}


In [7]:
for f in sorted(CAPTURES.rglob("conn.log")):
    if "bruteforce" not in f.parent.name:
        continue
    bf = pd.read_json(f, lines=True)
    if len(bf) < 50:
        print("skipping tiny capture:", f.parent.name, len(bf)); continue
    print("\n=====", f.parent.name, len(bf), "flows =====")
    print(bf["id.resp_p"].value_counts().head(5).to_string())
    if "conn_state" in bf.columns:
        print(bf["conn_state"].value_counts().head(6).to_string())
    num = [c for c in ["orig_pkts", "resp_pkts", "duration"] if c in bf.columns]
    print(bf[num].apply(pd.to_numeric, errors="coerce").describe().round(3))
    lab_bf = zeek_windows(bf)
    cic_bf = w[w["family"] == "bruteforce"]
    if len(lab_bf):
        print(pd.DataFrame({"CIC_bruteforce_median": cic_bf[FEATS].median(),
                            "lab_median": lab_bf[FEATS].median()}).round(3))


===== zeek_bruteforce_02 323 flows =====
id.resp_p
21     320
138      2
22       1
conn_state
SF      316
RSTR      4
S0        2
RSTO      1
       orig_pkts  resp_pkts  duration
count    323.000    323.000   322.000
mean      17.870     16.861     9.936
std        1.475      1.541     0.982
min        2.000      0.000     0.000
25%       18.000     17.000     9.602
50%       18.000     17.000     9.995
75%       18.000     17.000    10.389
max       21.000     20.000    11.261
                CIC_bruteforce_median  lab_median
n_dst_ports                      1.00         1.0
n_dst_hosts                      1.00         1.0
top_port_share                   1.00         1.0
no_reply_share                   0.10         0.0
mean_fwd_pkts                    5.85        18.0
mean_bwd_pkts                    8.50        17.0
skipping tiny capture: zeek_bruteforce_ssh_01 3
